# Warm-up 3B · Predict New Medicines: Molecule Binding · Dự đoán thuốc mới: phân tử có gắn với protein không? (Graph Learning · Học trên đồ thị)

| | 🇬🇧 English | 🇻🇳 Tiếng Việt |
|---|---|---|
| **Task** | Each row is a small molecule given as a **graph** (atoms = nodes, bonds = edges). Predict the probability that it binds to each of 3 protein targets: `bind_BRD4`, `bind_HSA`, `bind_sEH`. Binders are rare (a few %). | Mỗi dòng là một phân tử nhỏ được cho dưới dạng **đồ thị** (nguyên tử = đỉnh, liên kết = cạnh). Dự đoán xác suất phân tử gắn với từng protein đích: `bind_BRD4`, `bind_HSA`, `bind_sEH`. Phân tử gắn được rất hiếm (vài %). |
| **Data** | `train/` 40,000 molecules (with labels) · `public_test/` 5,000 · `private_test/` 5,000. Molecules are built from a core scaffold + 3 building blocks; some test molecules use building blocks or scaffolds that never appear in `train/`. | `train/` 40.000 phân tử (có nhãn) · `public_test/` 5.000 · `private_test/` 5.000. Phân tử được lắp từ một khung lõi + 3 khối xây dựng; một số phân tử trong tập test dùng khối xây dựng hoặc khung lõi không hề xuất hiện trong `train/`. |
| **Metric** | Mean Average Precision (mAP) over the 3 targets, **higher is better**. Ranking uses the **private** score. | Trung bình Average Precision (mAP) của 3 protein, **càng cao càng tốt**. Xếp hạng theo điểm **private**. |
| **Submission** | `public_submission.csv`, `private_submission.csv` with columns `id,bind_BRD4,bind_HSA,bind_sEH` (probabilities in [0, 1]) | `public_submission.csv`, `private_submission.csv` gồm cột `id,bind_BRD4,bind_HSA,bind_sEH` (xác suất trong [0, 1]) |
| **Rules** | PyTorch model, no external data, no chemistry libraries (e.g. RDKit). | Mô hình PyTorch, không dùng dữ liệu ngoài, không dùng thư viện hóa học (vd. RDKit). |

$$\text{AP}_t=\sum_{n}(R_n-R_{n-1})\,P_n \qquad \text{mAP}=\tfrac{1}{3}\left(\text{AP}_{\text{BRD4}}+\text{AP}_{\text{HSA}}+\text{AP}_{\text{sEH}}\right)$$

🇬🇧 $P_n$, $R_n$ = precision and recall when the $n$ highest-scored molecules are called binders (`sklearn.metrics.average_precision_score`). Only the ranking of your probabilities matters.
🇻🇳 $P_n$, $R_n$ = precision và recall khi coi $n$ phân tử có điểm cao nhất là "gắn được" (`sklearn.metrics.average_precision_score`). Chỉ thứ tự xếp hạng của xác suất là quan trọng.

**Columns · Các cột:**
- `id` — molecule id · mã phân tử
- `n_atoms` — number of heavy atoms · số nguyên tử nặng (hydrogens are implicit · nguyên tử H được lược bỏ)
- `atoms` — space-separated element symbols; atom *k* is the *k*-th symbol · ký hiệu nguyên tố cách nhau bởi dấu cách; nguyên tử thứ *k* là ký hiệu thứ *k*
- `bonds` — `i-j-t` separated by `;`, bond type `t` ∈ {`1` single, `2` double, `3` triple, `a` aromatic} · các liên kết `i-j-t` cách nhau bởi `;`, loại liên kết `t` ∈ {`1` đơn, `2` đôi, `3` ba, `a` thơm}
- `bind_BRD4, bind_HSA, bind_sEH` — targets (0/1), train only · nhãn (0/1), chỉ có ở train

Inspired by · Lấy cảm hứng từ: [NeurIPS 2024 – Leash Bio "Predict New Medicines with BELKA" (Kaggle, 2024)](https://www.kaggle.com/competitions/leash-BELKA).
The data here are synthetic · Dữ liệu ở đây là dữ liệu mô phỏng.

🇬🇧 This baseline runs end-to-end and submits a result. Beating it is up to you.
🇻🇳 Baseline này chạy trọn vẹn và nộp được kết quả. Vượt qua nó là việc của bạn.

## 1. Setup and Imports · Cài đặt và thư viện

In [ ]:
import os
import re
import shutil
import unicodedata
import zipfile
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.metrics import average_precision_score
from sklearn.model_selection import train_test_split

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 2. Configuration and Hyperparameters · Cấu hình và siêu tham số
🇬🇧 Nothing to fill in here: the data downloads itself, and your team comes from your account on the site when you submit (section 7).
🇻🇳 Không cần điền gì ở đây: dữ liệu tự tải về, và đội của bạn được lấy từ tài khoản trên trang web khi nộp bài (phần 7).


In [ ]:
# ---- Data / Dữ liệu ----
# The dataset downloads itself from GitHub straight onto this Colab session's own
# disk. No Google Drive and no mounting: reading thousands of small files over a
# Drive mount is what made the old flow slow, and local disk is several times
# faster per epoch.
# Bộ dữ liệu tự tải từ GitHub vào ổ đĩa của phiên Colab này. Không cần Drive.
TASK    = "3B"
FOLDER  = "3B_molecule_binding"
RELEASE = "https://github.com/ftulabs/olympiad-datasets/releases/download/warmup-3"

# Already have the folder? Point this at the parent of 3B_molecule_binding/ and nothing is
# downloaded. / Đã có sẵn thư mục? Trỏ vào thư mục cha của 3B_molecule_binding/.
ROOT_OVERRIDE = ""

import os, subprocess, time

ROOT = ROOT_OVERRIDE or "/content/olympiad"
if not os.path.isdir(f"{ROOT}/{FOLDER}/dataset"):
    os.makedirs(ROOT, exist_ok=True)
    zip_path = f"{ROOT}/{FOLDER}.zip"
    if not os.path.exists(zip_path):
        print(f"Downloading {FOLDER}.zip / Đang tải ...", flush=True)
        t0 = time.time()
        subprocess.run(["curl", "-fsSL", "--retry", "3", "-o", zip_path,
                        f"{RELEASE}/{FOLDER}.zip"], check=True)
        print(f"  {os.path.getsize(zip_path)/1e6:.0f} MB in {time.time()-t0:.0f}s", flush=True)
    print("Extracting / Đang giải nén ...", flush=True)
    subprocess.run(["unzip", "-nq", zip_path, "-d", ROOT], check=True)

if not os.path.isdir(f"{ROOT}/{FOLDER}/dataset"):
    raise FileNotFoundError(
        f"{ROOT}/{FOLDER}/dataset is missing after the download. Re-run this cell. "
        f"/ Thiếu dữ liệu sau khi tải. Hãy chạy lại ô này."
    )
print(f"Data root / Thư mục dữ liệu: {ROOT}")

DATA_DIR = f"{ROOT}/3B_molecule_binding/dataset"
TRAIN_CSV = f"{DATA_DIR}/train/train.csv"
PUBLIC_CSV = f"{DATA_DIR}/public_test/public_test.csv"
PRIVATE_CSV = f"{DATA_DIR}/private_test/private_test.csv"
RESULTS_DIR = f"{ROOT}/results"  # shared submission folder / thư mục nộp bài chung
SAVE_DIR = "./checkpoints"

# ---- Hyperparameters / Siêu tham số ----
BATCH_SIZE = 128
EPOCHS = 5
LR = 1e-3
HIDDEN = 64
SEED = 42

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

os.makedirs(SAVE_DIR, exist_ok=True)
np.random.seed(SEED)
torch.manual_seed(SEED)

## 3. Data Loading and Graph Batching · Đọc dữ liệu và ghép lô đồ thị

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
public_df = pd.read_csv(PUBLIC_CSV)
private_df = pd.read_csv(PRIVATE_CSV)
print(f"Train: {len(train_df)} | Public test: {len(public_df)} | Private test: {len(private_df)}")

TARGETS = ["bind_BRD4", "bind_HSA", "bind_sEH"]
print("Binder rate / Tỷ lệ gắn được:", train_df[TARGETS].mean().round(4).to_dict())

ATOM_TYPES = ["C", "N", "O", "S", "F", "Cl", "Br"]
ATOM_TO_IDX = {a: i for i, a in enumerate(ATOM_TYPES)}


def parse_graph(atoms, bonds):
    """EN: one CSV row -> atom indices + edge list (both directions).
    VI: một dòng CSV -> chỉ số loại nguyên tử + danh sách cạnh (cả hai chiều)."""
    x = torch.tensor([ATOM_TO_IDX[a] for a in atoms.split()], dtype=torch.long)
    pairs = [tuple(int(v) for v in b.split("-")[:2]) for b in bonds.split(";")]
    src = [i for i, j in pairs] + [j for i, j in pairs]
    dst = [j for i, j in pairs] + [i for i, j in pairs]
    return x, torch.tensor([src, dst], dtype=torch.long)


def make_graphs(df):
    """EN: list of (atoms, edges, labels); test rows get dummy zero labels.
    VI: danh sách (nguyên tử, cạnh, nhãn); dòng test nhận nhãn giả bằng 0."""
    labels = df[TARGETS].values if set(TARGETS) <= set(df.columns) else np.zeros((len(df), len(TARGETS)))
    graphs = []
    for atoms, bonds, y in zip(df["atoms"], df["bonds"], labels):
        x, edge_index = parse_graph(atoms, bonds)
        graphs.append((x, edge_index, torch.tensor(y, dtype=torch.float32)))
    return graphs


def collate(batch):
    """EN: merge many small graphs into one big disconnected graph; `batch_idx` tells which graph each atom belongs to.
    VI: ghép nhiều đồ thị nhỏ thành một đồ thị lớn rời rạc; `batch_idx` cho biết mỗi nguyên tử thuộc đồ thị nào."""
    xs, edges, ys, batch_idx = [], [], [], []
    offset = 0
    for g, (x, edge_index, y) in enumerate(batch):
        xs.append(x)
        edges.append(edge_index + offset)  # shift atom indices / dịch chỉ số nguyên tử
        ys.append(y)
        batch_idx.append(torch.full((len(x),), g, dtype=torch.long))
        offset += len(x)
    return torch.cat(xs), torch.cat(edges, dim=1), torch.cat(batch_idx), torch.stack(ys)


# EN: random 10% validation split / VI: tách ngẫu nhiên 10% làm tập validation
tr_df, val_df = train_test_split(train_df, test_size=0.1, random_state=SEED)
train_graphs, val_graphs = make_graphs(tr_df), make_graphs(val_df)
public_graphs, private_graphs = make_graphs(public_df), make_graphs(private_df)

train_loader = DataLoader(train_graphs, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)
val_loader = DataLoader(val_graphs, batch_size=512, shuffle=False, collate_fn=collate)
public_loader = DataLoader(public_graphs, batch_size=512, shuffle=False, collate_fn=collate)
private_loader = DataLoader(private_graphs, batch_size=512, shuffle=False, collate_fn=collate)
print(f"Train graphs: {len(train_graphs)} | Validation graphs: {len(val_graphs)}")
train_df.head()

## 4. Model Architecture · Kiến trúc mô hình

In [ ]:
class SimpleGNN(nn.Module):
    """EN: atom embedding -> 2 rounds of message passing (sum of neighbours) -> mean pooling -> 3 logits.
    VI: embedding nguyên tử -> 2 vòng truyền thông điệp (tổng các láng giềng) -> gộp trung bình -> 3 logit."""

    def __init__(self, n_atom_types, hidden=HIDDEN, n_out=3):
        super().__init__()
        self.embed = nn.Embedding(n_atom_types, hidden)
        self.mp1 = nn.Linear(2 * hidden, hidden)
        self.mp2 = nn.Linear(2 * hidden, hidden)
        self.head = nn.Linear(hidden, n_out)

    def message_passing(self, h, edge_index, layer):
        src, dst = edge_index
        msg = torch.zeros_like(h).index_add_(0, dst, h[src])  # EN: sum over neighbours / VI: cộng các láng giềng
        return torch.relu(layer(torch.cat([h, msg], dim=1)))

    def forward(self, x, edge_index, batch_idx, n_graphs):
        h = self.embed(x)
        h = self.message_passing(h, edge_index, self.mp1)
        h = self.message_passing(h, edge_index, self.mp2)
        # EN: mean pooling = sum of atom vectors / number of atoms / VI: gộp trung bình = tổng vector / số nguyên tử
        pooled = torch.zeros(n_graphs, h.size(1), device=h.device).index_add_(0, batch_idx, h)
        counts = torch.zeros(n_graphs, device=h.device).index_add_(0, batch_idx, torch.ones_like(h[:, 0]))
        return self.head(pooled / counts.unsqueeze(1))


model = SimpleGNN(len(ATOM_TYPES)).to(DEVICE)
print(model)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters()):,}")

## 5. Training · Huấn luyện

In [ ]:
def run_batch(model, batch):
    x, edge_index, batch_idx, y = (t.to(DEVICE) for t in batch)
    return model(x, edge_index, batch_idx, len(y)), y


@torch.no_grad()
def predict_proba(model, loader):
    """EN: probabilities, shape (n_molecules, 3). / VI: xác suất, kích thước (số phân tử, 3)."""
    model.eval()
    return np.concatenate([torch.sigmoid(run_batch(model, b)[0]).cpu().numpy() for b in loader])


def mean_ap(y_true, y_prob):
    return float(np.mean([average_precision_score(y_true[:, k], y_prob[:, k]) for k in range(y_true.shape[1])]))


criterion = nn.BCEWithLogitsLoss()  # model outputs logits / mô hình trả về logit
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
y_val = val_df[TARGETS].values

best_map = -1.0
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for batch in train_loader:
        optimizer.zero_grad()
        logits, y = run_batch(model, batch)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y)

    val_map = mean_ap(y_val, predict_proba(model, val_loader))
    msg = f"Epoch {epoch:02d} | Loss={total_loss / len(train_graphs):.4f} | Val mAP={val_map:.4f}"
    if val_map > best_map:
        best_map = val_map
        torch.save(model.state_dict(), os.path.join(SAVE_DIR, "best_model.pt"))
        msg += "  ✓ best model saved"
    print(msg)

print(f"\nBest validation mAP: {best_map:.4f}")

## 6. Predict Public & Private Test · Dự đoán tập Public & Private
🇬🇧 Creates `public_submission.csv` / `private_submission.csv` (+ `.zip`), like the official competition.
🇻🇳 Tạo `public_submission.csv` / `private_submission.csv` (+ `.zip`), giống định dạng cuộc thi chính thức.

In [ ]:
def export(ids, probs, csv_path):
    """Save <id, bind_BRD4, bind_HSA, bind_sEH> CSV and zip it. / Lưu dự đoán ra CSV và nén zip."""
    sub = pd.DataFrame(probs, columns=TARGETS)
    sub.insert(0, "id", ids.values)
    sub.to_csv(csv_path, index=False)
    zip_path = csv_path.replace(".csv", ".zip")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(csv_path, arcname=os.path.basename(csv_path))
    print(f"Saved {csv_path} and {zip_path}")


model.load_state_dict(torch.load(os.path.join(SAVE_DIR, "best_model.pt"), map_location=DEVICE))
export(public_df["id"], predict_proba(model, public_loader), "public_submission.csv")
export(private_df["id"], predict_proba(model, private_loader), "private_submission.csv")
pd.read_csv("public_submission.csv").head()

## 7. Submit · Nộp bài

🇬🇧 Your team comes from your account on [seleccion.ftds.online](https://seleccion.ftds.online/), not from a name typed here. The cell below prints a short code once per Colab session; type it on the site and this notebook submits as you.

🇻🇳 Đội của bạn lấy từ tài khoản trên [seleccion.ftds.online](https://seleccion.ftds.online/). Ô bên dưới in ra một mã ngắn mỗi phiên Colab; nhập mã đó trên trang web.


In [ ]:
def check_submission(csv_path, test_df):
    """Validate the file format before submitting. / Kiểm tra định dạng file trước khi nộp."""
    sub = pd.read_csv(csv_path)
    assert list(sub.columns) == ["id", *TARGETS], f"{csv_path}: columns must be {['id', *TARGETS]}"
    assert len(sub) == len(test_df) and set(sub["id"]) == set(test_df["id"]), f"{csv_path}: ids do not match the test set"
    assert sub[TARGETS].notna().all().all(), f"{csv_path}: contains missing values / có giá trị bị thiếu"
    assert sub[TARGETS].min().min() >= 0 and sub[TARGETS].max().max() <= 1, f"{csv_path}: probabilities must be in [0, 1]"


check_submission("public_submission.csv", public_df)
check_submission("private_submission.csv", private_df)


In [ ]:
# ===================================================================
#  Olympiad AI - submit.  Paste at the END of your notebook, then run.
#  No key to copy: the first run prints a code you enter on the site.
# ===================================================================
BASE    = "https://seleccion.ftds.online"
PROBLEM = "3B"          # which problem this notebook is for
CONTEST = ""            # blank = the contest you are entered in
PUBLIC  = "public_submission.csv"
PRIVATE = "private_submission.csv"
# -------------------------------------------------------------------
import os, time, requests

token = globals().get("_OLY_TOKEN")
if not token:
    start = requests.post(f"{BASE}/api/pair/start", timeout=30).json()
    print(f"\n  1. open   {start['enter_at']}")
    print(f"  2. enter  {start['code']}\n  waiting...", flush=True)
    for _ in range(int(start["expires_in"]) // 3):
        time.sleep(3)
        got = requests.get(f"{BASE}/api/pair/poll",
                           params={"code": start["code"]}, timeout=30).json()
        if got.get("ready"):
            token = globals()["_OLY_TOKEN"] = got["token"]
            print("  paired ✓ (this notebook stays paired for the session)")
            break
    else:
        raise SystemExit("pairing timed out - run this cell again")

missing = [f for f in (PUBLIC, PRIVATE) if not os.path.exists(f)]
if PUBLIC in missing:
    raise SystemExit(f"{PUBLIC} not found - run the cell that writes it first")

files = {"public": open(PUBLIC, "rb")}
if PRIVATE not in missing:
    files["private"] = open(PRIVATE, "rb")
try:
    r = requests.post(f"{BASE}/api/submit", headers={"X-Oly-Token": token},
                      data={"problem": PROBLEM, "contest": CONTEST},
                      files=files, timeout=600)
finally:
    for f in files.values():
        f.close()

try:
    d = r.json()
except ValueError:
    raise SystemExit(f"server {r.status_code}: {r.text[:300]}")

if r.status_code == 429:
    print(f"⏳ cooldown - wait {d.get('retry_after')}s and run again")
elif r.status_code == 401:
    globals().pop("_OLY_TOKEN", None)
    print(f"❌ {d.get('detail')}  (run this cell again to re-pair)")
elif r.status_code != 200:
    print(f"❌ {r.status_code}: {d.get('detail') or d.get('message')}")
elif d["public"]["status"] != "ok":
    # the grader's own message about your file - it says what to fix
    print(f"❌ {d['public']['status'].upper()}: {d['public']['message']}")
else:
    p, c = d["public"], (d.get("counting") or {})
    print(f"✅ {d['problem']} · {d['title']}")
    print(f"   {d['metric']} = {p['score']:.4f}   {p['tier'] or '(below the lowest tier)'}")
    print(f"   entry #{d['entries']}   ·   rank: {d['rank']}")
    if c.get("score") is not None and abs(c["score"] - p["score"]) > 1e-12:
        print(f"   counting entry: {c['score']:.4f} {c['tier']}")
    print(f"   {BASE}/")
